# 08 — Model Evaluation

## Goal
Honest test-set comparison: ROC-AUC, PR-AUC, F1, confusion matrix, threshold choice and probability calibration.

In [ ]:
from pathlib import Path
import sys

# Locate the project root (works whether the notebook is run from notebooks/ or root)
root = Path.cwd()
while not (root / "src" / "data_loader.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
print("project root:", root)


In [ ]:
from pathlib import Path

missing = [p.name for p in [
    root / "models" / "metadata.json",
    root / "reports" / "model_results.csv",
] if not p.exists()]
if missing:
    print("Missing artifacts:", missing)
    print("Run first:  python -m src.train && python -m src.evaluate && python -m src.explain")
else:
    print("All artifacts present.")


In [ ]:
pd.read_csv(root / "reports" / "model_results.csv")


In [ ]:
import json, matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import joblib
from src.train import prepare_matrices

meta = json.loads((root / "models" / "metadata.json").read_text())
ev = json.loads((root / "reports" / "eval_summary.json").read_text())
data = prepare_matrices()
model = joblib.load(root / "models" / "xgboost_calibrated.joblib")
pred = (model.predict_proba(data["X_test"])[:, 1] >= ev["threshold"]).astype(int)
cm = confusion_matrix(data["y_test"], pred)
fig, ax = plt.subplots(figsize=(4.5, 4), constrained_layout=True)
ConfusionMatrixDisplay(cm, display_labels=["No default", "Default"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"XGBoost calibrated @ threshold {ev['threshold']:.2f}")
plt.show()


In [ ]:
print(open(root / "reports" / "classification_report.txt").read())


**Business reading** — FN = missed defaulter (direct loss exposure), FP = good client wrongly flagged. At threshold 0.26 the model catches 58% of defaulters at 51% precision. Calibration: isotonic gives the lowest Brier score; the reliability curve hugs the diagonal.